In [1]:
# connect 4 witht ai 

In [1]:
import pygame
import sys
import numpy as np
import math
import random


ROW_COUNT = 6
COLUMN_COUNT = 7

BLUE = (0, 0, 255)
BLACK = (0, 0, 0)
RED = (255, 0, 0)
YELLOW = (255, 255, 0)

PLAYER = 0
AI = 1

PLAYER_PIECE = 1
AI_PIECE = 2
EMPTY = 0

WINDOW_LENGTH = 4
SQUARESIZE = 100
RADIUS = int(SQUARESIZE / 2 - 5)

width = COLUMN_COUNT * SQUARESIZE
height = (ROW_COUNT + 1) * SQUARESIZE
size = (width, height)


def create_board():
    return np.zeros((ROW_COUNT, COLUMN_COUNT))

def drop_piece(board, row, col, piece):
    board[row][col] = piece

def is_valid_location(board, col):
    return board[ROW_COUNT - 1][col] == 0

def get_next_open_row(board, col):
    for r in range(ROW_COUNT):
        if board[r][col] == 0:
            return r

def get_valid_locations(board):
    return [c for c in range(COLUMN_COUNT) if is_valid_location(board, c)]

def winning_move(board, piece):
    for c in range(COLUMN_COUNT - 3):
        for r in range(ROW_COUNT):
            if all(board[r][c + i] == piece for i in range(4)):
                return True

    for c in range(COLUMN_COUNT):
        for r in range(ROW_COUNT - 3):
            if all(board[r + i][c] == piece for i in range(4)):
                return True

    for c in range(COLUMN_COUNT - 3):
        for r in range(ROW_COUNT - 3):
            if all(board[r + i][c + i] == piece for i in range(4)):
                return True

    for c in range(COLUMN_COUNT - 3):
        for r in range(3, ROW_COUNT):
            if all(board[r - i][c + i] == piece for i in range(4)):
                return True

    return False


def evaluate_window(window, piece):
    score = 0
    opp_piece = PLAYER_PIECE if piece == AI_PIECE else AI_PIECE

    if window.count(piece) == 4:
        score += 100
    elif window.count(piece) == 3 and window.count(EMPTY) == 1:
        score += 5
    elif window.count(piece) == 2 and window.count(EMPTY) == 2:
        score += 2

    if window.count(opp_piece) == 3 and window.count(EMPTY) == 1:
        score -= 4

    return score

def score_position(board, piece):
    score = 0

    center_array = [int(i) for i in board[:, COLUMN_COUNT // 2]]
    score += center_array.count(piece) * 3

    for r in range(ROW_COUNT):
        row_array = [int(i) for i in board[r, :]]
        for c in range(COLUMN_COUNT - 3):
            score += evaluate_window(row_array[c:c + WINDOW_LENGTH], piece)

    for c in range(COLUMN_COUNT):
        col_array = [int(i) for i in board[:, c]]
        for r in range(ROW_COUNT - 3):
            score += evaluate_window(col_array[r:r + WINDOW_LENGTH], piece)

    for r in range(ROW_COUNT - 3):
        for c in range(COLUMN_COUNT - 3):
            score += evaluate_window([board[r + i][c + i] for i in range(4)], piece)

    for r in range(3, ROW_COUNT):
        for c in range(COLUMN_COUNT - 3):
            score += evaluate_window([board[r - i][c + i] for i in range(4)], piece)

    return score

def is_terminal_node(board):
    return (winning_move(board, PLAYER_PIECE) or
            winning_move(board, AI_PIECE) or
            len(get_valid_locations(board)) == 0)


def minimax(board, depth, alpha, beta, maximizing):
    valid_locations = get_valid_locations(board)
    terminal = is_terminal_node(board)

    if depth == 0 or terminal:
        if terminal:
            if winning_move(board, AI_PIECE):
                return None, 10**14
            elif winning_move(board, PLAYER_PIECE):
                return None, -10**14
            else:
                return None, 0
        else:
            return None, score_position(board, AI_PIECE)

    if maximizing:
        value = -math.inf
        best_col = random.choice(valid_locations)
        for col in valid_locations:
            row = get_next_open_row(board, col)
            temp = board.copy()
            drop_piece(temp, row, col, AI_PIECE)
            new_score = minimax(temp, depth - 1, alpha, beta, False)[1]
            if new_score > value:
                value = new_score
                best_col = col
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        return best_col, value

    else:
        value = math.inf
        best_col = random.choice(valid_locations)
        for col in valid_locations:
            row = get_next_open_row(board, col)
            temp = board.copy()
            drop_piece(temp, row, col, PLAYER_PIECE)
            new_score = minimax(temp, depth - 1, alpha, beta, True)[1]
            if new_score < value:
                value = new_score
                best_col = col
            beta = min(beta, value)
            if alpha >= beta:
                break
        return best_col, value


def draw_board(board):
    for c in range(COLUMN_COUNT):
        for r in range(ROW_COUNT):
            pygame.draw.rect(screen, BLUE,
                (c * SQUARESIZE, r * SQUARESIZE + SQUARESIZE, SQUARESIZE, SQUARESIZE))
            pygame.draw.circle(screen, BLACK,
                (c * SQUARESIZE + SQUARESIZE // 2,
                 r * SQUARESIZE + SQUARESIZE + SQUARESIZE // 2), RADIUS)

    for c in range(COLUMN_COUNT):
        for r in range(ROW_COUNT):
            if board[r][c] == PLAYER_PIECE:
                pygame.draw.circle(screen, RED,
                    (c * SQUARESIZE + SQUARESIZE // 2,
                     height - (r * SQUARESIZE + SQUARESIZE // 2)), RADIUS)
            elif board[r][c] == AI_PIECE:
                pygame.draw.circle(screen, YELLOW,
                    (c * SQUARESIZE + SQUARESIZE // 2,
                     height - (r * SQUARESIZE + SQUARESIZE // 2)), RADIUS)

    pygame.display.update()


pygame.init()
screen = pygame.display.set_mode(size)
pygame.display.set_caption("Connect 4 – Human vs AI")
font = pygame.font.SysFont("monospace", 75)

board = create_board()
turn = PLAYER
game_over = False

draw_board(board)

while not game_over:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            sys.exit()

        if event.type == pygame.MOUSEMOTION and turn == PLAYER:
            pygame.draw.rect(screen, BLACK, (0, 0, width, SQUARESIZE))
            pygame.draw.circle(screen, RED, (event.pos[0], SQUARESIZE // 2), RADIUS)
            pygame.display.update()

        if event.type == pygame.MOUSEBUTTONDOWN and turn == PLAYER:
            col = event.pos[0] // SQUARESIZE
            if is_valid_location(board, col):
                row = get_next_open_row(board, col)
                drop_piece(board, row, col, PLAYER_PIECE)

                if winning_move(board, PLAYER_PIECE):
                    screen.blit(font.render("You Win!", True, RED), (40, 10))
                    game_over = True

                turn = AI
                draw_board(board)

    if turn == AI and not game_over:
        col, _ = minimax(board, 4, -math.inf, math.inf, True)
        row = get_next_open_row(board, col)
        drop_piece(board, row, col, AI_PIECE)

        if winning_move(board, AI_PIECE):
            screen.blit(font.render("AI Wins!", True, YELLOW), (40, 10))
            game_over = True

        turn = PLAYER
        draw_board(board)

    if game_over:
        pygame.time.wait(3000)


pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html
